<a href="https://colab.research.google.com/github/KatreenGhobrial/RepoCloudComputing/blob/main/Tirgulim/Tirgul4/3_1_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
{
  "variable-1": 1.0,
  "variable-2": 4.5,
  "variable-3": 2.3,
  "variable-4": 1.4,
  "variable-5": 1.3
}


{'variable-1': 1.0,
 'variable-2': 4.5,
 'variable-3': 2.3,
 'variable-4': 1.4,
 'variable-5': 1.3}

In [5]:
url = 'https://data.gov.il/api/3/action/datastore_search?resource_id=053cea08-09bc-40ec-8f7a-156f0677aff3'

import requests
import pandas as pd
from ipywidgets import widgets, VBox, Output
from IPython.display import display

pd.set_option("display.max_columns", None)

response = requests.get(url)
data = response.json()
df = pd.DataFrame(data)
data_df = pd.DataFrame(data['result']['records'])
data_df.head()
output_area = Output()

# Create widgets
tozeret_nm_dropdown = widgets.Dropdown(
    options=[''] + sorted(data_df['tozeret_nm'].unique().tolist()),
    description='Tozeret:',
    style={'description_width': 'initial'}
)

kinuy_mishari_dropdown = widgets.Dropdown(
    options=[''],
    description='Kinuy Mishari:',
    style={'description_width': 'initial'}
)
# Function to update the second dropdown based on the selection of the first
def update_kinuy_mishari_options(change):
    if change['new']:  # Check if a valid option is selected
        filtered_values = data_df[data_df['tozeret_nm'] == change['new']]['kinuy_mishari'].unique()
        kinuy_mishari_dropdown.options = [''] + sorted(filtered_values)
    else:
        kinuy_mishari_dropdown.options = ['']

# Function to update the output area with the total records and unique 'ramat_gimur' values
def update_output(change=None):
    output_area.clear_output()
    selected_tozeret = tozeret_nm_dropdown.value
    selected_kinuy = kinuy_mishari_dropdown.value

    if selected_tozeret and selected_kinuy:
            filtered_df = data_df[
                (data_df['tozeret_nm'] == selected_tozeret) &
                (data_df['kinuy_mishari'] == selected_kinuy)
            ]
            total_records = len(filtered_df)
            unique_ramat_gimur = filtered_df['ramat_gimur'].unique()

            with output_area:
                    print(f"Total Records: {total_records}")
                    print(f"Unique Ramat Gimur: {', '.join(unique_ramat_gimur)
                    if 							len(unique_ramat_gimur) > 0
                          else 'None'}")
    else:
          with output_area:
              print("Please select valid options for both dropdowns.")


# Observe changes in both dropdowns
tozeret_nm_dropdown.observe(update_kinuy_mishari_options, names='value')

kinuy_mishari_dropdown.observe(update_output, names='value')

# Display widgets and output area
display(VBox([tozeret_nm_dropdown, kinuy_mishari_dropdown, output_area]))





In [20]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from plotly.offline import iplot
import plotly.graph_objs as go
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns  #seaborn for heatmap


# Create a simple DataFrame
data = {
    'Name': ['Alice', 'Bob', 'Charlie'],
    'Age': [25, 30, 35],
    'City': ['New York', 'Los Angeles', 'Chicago']
}
df = pd.DataFrame(data)

# Tab 1: Data Overview
tab1_content = widgets.Output()
with tab1_content:
    print("statistics overview")
    display(data_df.describe())



# Tab 2: Raw Data
tab2_content = widgets.Output()
with tab2_content:
    print("Raw Data:")
    display(data_df)  # Full DataFrame

# Tab 3: Charts
tab3_content = widgets.Output()
with tab3_content:
    xAxis=data_df['tozeret_nm']
    yAxis=data_df['shnat_yitzur']

    plt.figure(figsize=(12, 6))
    plt.plot(xAxis, yAxis, marker='o', linestyle='-')
    #plt.title('Enhanced Heatmap with Labels')
    plt.xlabel('X Axis Cars')
    plt.ylabel('Y Axis Years')
    plt.show()  # Add this to display the plot
    plt.close()  # Close the figure t free memory
    # print("Charts:")
    # plt.clf()  # Clear the current figure
    # plt.figure(figsize=(10, 8))  # Increased figure size to accommodate labels
    # side_length = 10
    # data = 5 + np.random.randn(side_length, side_length)
    # data += np.arange(side_length)
    # data += np.reshape(np.arange(side_length), (side_length, 1))

    # # Create heatmap with labels
    # # heatmap = sns.heatmap(data,
    # #                      cmap='viridis',
    # #                      annot=True,
    # #                      fmt='.1f',
    # #                      cbar_kws={'label': 'Value Intensity'})
    # xAxis=data_df['tozeret_nm']
    # yAxis=data_df['shnat_yitzur']
    # plt.plot(xAxis, yAxis, marker='o', linestyle='-')
    # #plt.title('Enhanced Heatmap with Labels')
    # plt.xlabel('X Axis Cars')
    # plt.ylabel('Y Axis Years')
    # plt.show()  # Add this to display the plot
    # plt.close()  # Close the figure t free memory



# Create Tabs
tabs = widgets.Tab(children=[tab1_content, tab2_content,tab3_content])
tabs.set_title(0, 'Tab 1: Data Overview')
tabs.set_title(1, 'Tab 2: Raw Data')
tabs.set_title(2, 'Tab 3: Charts')



# Display Tabs
display(tabs)


In [ ]:
import requests
import pandas as pd
import plotly.graph_objects as go

# 1. Fetch the data from the API
url = 'https://data.gov.il/api/3/action/datastore_search'
params = {
    'resource_id': '053cea08-09bc-40ec-8f7a-156f0677aff3',
    'limit': 1000 # Fetching 1000 records for a good sample
}
response = requests.get(url, params=params)
data = response.json()
df = pd.DataFrame(data['result']['records'])

# 2. Aggregate the data: Count vehicles per manufacturer ('tozeret_nm')
# This gives us unique manufacturers and their respective counts
manufacturer_counts = df['tozeret_nm'].value_counts().reset_index()
manufacturer_counts.columns = ['tozeret_nm', 'count']

# 3. Prepare the data for the Sunburst chart
# We need a "Root" node that holds everything together
root_label = "All Vehicles"

# Labels: The root node + all unique manufacturer names
labels = [root_label] + manufacturer_counts['tozeret_nm'].tolist()

# Parents: The root has no parent (""). All manufacturers have the root as their parent.
parents = [""] + [root_label] * len(manufacturer_counts)

# Values: The total sum for the root + the individual counts for manufacturers
values = [manufacturer_counts['count'].sum()] + manufacturer_counts['count'].tolist()

# 4. Create the Figure
fig = go.Figure()

fig.add_trace(go.Sunburst(
    labels=labels,
    parents=parents,
    values=values,
    branchvalues="total" # Ensures the parent size matches the sum of its children
))

fig.update_layout(
    margin=dict(t=0, l=0, r=0, b=0)
)

fig.show()